# 11. EKF-SLAM

SLAM에서는 로봇 pose와 landmark 위치를 모두 상태에 넣는다.

$$x=[x_r,y_r,\theta_r,m_{1x},m_{1y},\dots,m_{Nx},m_{Ny}]^T$$

EKF-SLAM의 중요한 점은 landmark들이 관측을 공유하면서 서로 상관관계를 갖는다는 것이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 작은 EKF-SLAM 예제

랜드마크 3개를 이미 초기화했다고 가정하고, range-bearing 업데이트가 pose와 landmark를 함께 보정하는 모습을 확인한다.

In [ ]:
np.random.seed(23)
def wrap(a): return np.arctan2(np.sin(a),np.cos(a))
landmarks_true=np.array([[2.0,1.0],[5.0,1.5],[4.0,4.0]])
N=len(landmarks_true); dim=3+2*N
mu=np.zeros(dim); mu[:3]=[0.2,-0.1,0.1]
mu[3:]= (landmarks_true + np.random.randn(N,2)*0.35).ravel()
P=np.eye(dim)*0.15; P[:3,:3]=np.diag([0.2,0.2,0.08])
true_pose=np.array([0.0,0.0,0.0])
Q=np.diag([0.12**2,np.deg2rad(4)**2])
poses=[]; estposes=[]
controls=[(0.45,0.18)]*16+[(0.45,-0.12)]*18+[(0.35,0.1)]*16

def motion_pose(x,u,dt=0.2):
    v,w=u; return np.array([x[0]+v*dt*np.cos(x[2]), x[1]+v*dt*np.sin(x[2]), wrap(x[2]+w*dt)])
def h_full(mu,j):
    idx=3+2*j; mx,my=mu[idx:idx+2]; x,y,th=mu[:3]
    dx=mx-x; dy=my-y
    return np.array([np.hypot(dx,dy), wrap(np.arctan2(dy,dx)-th)])
def H_full(mu,j):
    idx=3+2*j; mx,my=mu[idx:idx+2]; x,y,th=mu[:3]
    dx=mx-x; dy=my-y; q=dx*dx+dy*dy; r=np.sqrt(q)
    H=np.zeros((2,dim))
    H[:,0:3]=np.array([[-dx/r,-dy/r,0],[dy/q,-dx/q,-1]])
    H[:,idx:idx+2]=np.array([[dx/r,dy/r],[-dy/q,dx/q]])
    return H

for u in controls:
    true_pose=motion_pose(true_pose,u)
    # predict pose only
    th=mu[2]; v,w=u; dt=0.2
    G=np.eye(dim); G[0,2]=-v*dt*np.sin(th); G[1,2]=v*dt*np.cos(th)
    mu[:3]=motion_pose(mu[:3],u,dt)
    R3=np.diag([0.03,0.03,0.01])
    P=G@P@G.T; P[:3,:3]+=R3
    # update all landmarks
    for j,m in enumerate(landmarks_true):
        dx=m[0]-true_pose[0]; dy=m[1]-true_pose[1]
        z=np.array([np.hypot(dx,dy), wrap(np.arctan2(dy,dx)-true_pose[2])]) + np.random.multivariate_normal([0,0],Q)
        H=H_full(mu,j); innov=z-h_full(mu,j); innov[1]=wrap(innov[1])
        S=H@P@H.T+Q; K=P@H.T@np.linalg.inv(S)
        mu=mu+K@innov; mu[2]=wrap(mu[2])
        P=(np.eye(dim)-K@H)@P
    poses.append(true_pose.copy()); estposes.append(mu[:3].copy())
poses=np.array(poses); estposes=np.array(estposes); lm_est=mu[3:].reshape(N,2)

fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].plot(poses[:,0],poses[:,1],'k-',lw=2,label='true robot')
axes[0].plot(estposes[:,0],estposes[:,1],color='#E85D24',lw=2,label='EKF-SLAM robot')
axes[0].scatter(landmarks_true[:,0],landmarks_true[:,1],marker='*',s=160,color='#1D9E75',label='true landmarks')
axes[0].scatter(lm_est[:,0],lm_est[:,1],s=90,color='#534AB7',label='estimated landmarks')
axes[0].axis('equal'); axes[0].grid(alpha=0.25); axes[0].legend(); axes[0].set_title('EKF-SLAM state estimate')
C=P.copy(); axes[1].imshow(np.abs(C),cmap='magma'); axes[1].set_title('absolute covariance matrix: correlations')
plt.tight_layout(); plt.savefig('assets/11_ekf_slam.png',dpi=150,bbox_inches='tight'); plt.show()
print('robot final error:', np.linalg.norm(estposes[-1,:2]-poses[-1,:2]).round(4))
print('landmark errors:', np.round(np.linalg.norm(lm_est-landmarks_true,axis=1),4))

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Joint state | robot pose + landmarks | Ch.10 SLAM |
| Cross covariance | landmark 간 상관관계 | EKF-SLAM의 핵심 성질 |
| Data association | 어떤 관측이 어떤 landmark인지 | 실제 SLAM의 난점 |